In [1]:
!pip install multilingual-clip torch
!pip install transformers==4.49
!pip install diffusers==0.32.2
#downgrade transformers and diffusers to avoid clip text error https://github.com/huggingface/diffusers/issues/12436

In [2]:
from transformers import ClapProcessor, ClapAudioModel
from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image
from torch.utils.data import Dataset, DataLoader
import torch
import time
import librosa
import torch.nn.functional as F
import torch.nn as nn
from PIL import Image
import numpy as np
from multilingual_clip import pt_multilingual_clip
import transformers
import types

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using devices: {device}")

using devices: cuda


In [11]:
#monkey patch function to get embeddings before linear pooling layer
#source code: https://github.com/FreddeFrallan/Multilingual-CLIP/blob/main/multilingual_clip/pt_multilingual_clip.py
def get_last_hidden_state(self, txt, tokenizer, device):
    txt_tok = tokenizer(txt, padding="max_length", truncation=True, max_length=77, return_tensors='pt').to(device)
    outputs = self.transformer(**txt_tok)[0]
    return outputs


In [12]:
multi_lingual_clip_name = 'M-CLIP/XLM-Roberta-Large-Vit-L-14'

# Load Model & Tokenizer
mclip_model = pt_multilingual_clip.MultilingualCLIP.from_pretrained(multi_lingual_clip_name).to(device)
#apply patch
mclip_model.get_last_hidden_state = types.MethodType(get_last_hidden_state, mclip_model)

mclip_tokenizer = transformers.AutoTokenizer.from_pretrained(multi_lingual_clip_name)

mclip_model.eval()

MultilingualCLIP(
  (transformer): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024

In [5]:
#sd_model_name = "stable-diffusion-v1-5/stable-diffusion-v1-5" #768 token embed dim
sd_model_name = "stabilityai/sd-turbo" #1024 token embed dim
sd_model = StableDiffusionPipeline.from_pretrained(sd_model_name, torch_dtype=torch.float16).to(device)
sd_text_encoder = sd_model.text_encoder
sd_tokenizer = sd_model.tokenizer
sd_text_encoder.eval()

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 1024)
      (position_embedding): Embedding(77, 1024)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-22): 23 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1

Section: Training linear projection layer to map varied token size from multi-lingual clip to fixed size 77

In [6]:
class CLIP2SDAdapter(torch.nn.Module):
    #keep in_tokens fixed at 16, arbitrary choice
    def __init__(self, in_dim=1024, out_dim=1024, in_tokens=77, out_tokens=77):
        super().__init__()
        self.proj = torch.nn.Linear(in_dim, out_dim)
        self.token_compressor = torch.nn.Linear(in_tokens, out_tokens)

    def forward(self, x):
        # x: [batch, in_tokens, in_dim]
        x = self.proj(x)
        # transpose to [batch, in_dim, in_tokens] → compress → back
        x = self.token_compressor(x.transpose(1, 2)).transpose(1, 2)
        return x  # [batch, 77, 1024]


In [7]:
text_pairs = {
    "a cat sleeping on the sofa": "un gato durmiendo en el sofá",
    "a cup of coffee on the table": "una taza de café sobre la mesa",
    "the mountain covered in snow": "la montaña cubierta de nieve",
    "a red sports car parked outside": "un coche deportivo rojo estacionado afuera",
    "children playing football": "niños jugando al fútbol",
    "a sunset over the ocean": "una puesta de sol sobre el océano",
    "a bowl of fresh fruit": "un cuenco de fruta fresca",
    "a person riding a bicycle": "una persona montando en bicicleta",
    "a dog running in the park": "un perro corriendo en el parque",
    "a busy city street at night": "una calle de ciudad concurrida por la noche",
    "a steaming bowl of soup": "un cuenco humeante de sopa",
    "a book open on a desk": "un libro abierto sobre un escritorio",
    "a plane flying above the clouds": "un avión volando sobre las nubes",
    "a bouquet of colorful flowers": "un ramo de flores coloridas",
    "a cozy cabin in the forest": "una cabaña acogedora en el bosque",
    "a smiling woman holding an umbrella": "una mujer sonriente sosteniendo un paraguas",
    "a man playing the guitar": "un hombre tocando la guitarra",
    "a chef preparing sushi": "un chef preparando sushi",
    "a lighthouse near the sea": "un faro cerca del mar",
    "a train passing through the countryside": "un tren pasando por el campo",
    "a person reading under a tree": "una persona leyendo bajo un árbol",
    "a glass of red wine": "una copa de vino tinto",
    "a soccer stadium full of fans": "un estadio de fútbol lleno de aficionados",
    "a forest path covered in leaves": "un sendero del bosque cubierto de hojas",
    "a horse running on the beach": "un caballo corriendo en la playa",
    "a plate of spaghetti": "un plato de espaguetis",
    "a candle lighting a dark room": "una vela iluminando una habitación oscura",
    "a person painting a landscape": "una persona pintando un paisaje",
    "a couple walking in the rain": "una pareja caminando bajo la lluvia",
    "a baby laughing": "un bebé riendo",
    "a street market with fresh vegetables": "un mercado callejero con verduras frescas",
    "a man wearing sunglasses": "un hombre con gafas de sol",
    "a smartphone on a wooden table": "un teléfono inteligente sobre una mesa de madera",
    "a dolphin jumping out of the water": "un delfín saltando fuera del agua",
    "a mountain road full of curves": "una carretera de montaña llena de curvas",
    "a violin resting on a chair": "un violín descansando en una silla",
    "a plate of tacos": "un plato de tacos",
    "a woman drinking tea": "una mujer bebiendo té",
    "a lion resting in the savanna": "un león descansando en la sabana",
    "a computer with colorful lights": "un ordenador con luces de colores",
    "a painter mixing colors": "un pintor mezclando colores",
    "a person climbing a mountain": "una persona escalando una montaña",
    "a coffee shop on a corner": "una cafetería en una esquina",
    "a small boat on the lake": "un pequeño barco en el lago",
    "a baker making bread": "un panadero haciendo pan",
    "a robot cleaning the floor": "un robot limpiando el suelo",
    "a car driving through the desert": "un coche conduciendo por el desierto",
    "a butterfly sitting on a flower": "una mariposa posada sobre una flor",
    "a beach full of umbrellas": "una playa llena de sombrillas",
    "a waiter serving food": "un camarero sirviendo comida",
    "a plate of sushi": "un plato de sushi",
    "a bowl of ramen": "un cuenco de ramen",
    "freshly baked bread": "pan recién horneado",
    "a slice of chocolate cake": "una porción de pastel de chocolate",
    "a basket of strawberries": "una cesta de fresas",
    "a pizza with cheese and olives": "una pizza con queso y aceitunas",
    "grilled salmon with lemon": "salmón a la parrilla con limón",
    "a glass of orange juice": "un vaso de jugo de naranja",
    "pasta with tomato sauce": "pasta con salsa de tomate",
    "a table full of desserts": "una mesa llena de postres",
    "a chef cutting vegetables": "un chef cortando verduras",
    "a cup of green tea": "una taza de té verde",
    "a sandwich with ham and cheese": "un sándwich con jamón y queso",
    "a bowl of cereal with milk": "un cuenco de cereales con leche",
    "spices on a wooden table": "especias sobre una mesa de madera",
    "a pot of hot soup": "una olla de sopa caliente",
    "a croissant and coffee": "un croissant y café",
    "a tray of sushi rolls": "una bandeja de rollos de sushi",
    "a salad with avocado": "una ensalada con aguacate",
    "a taco stand in the street": "un puesto de tacos en la calle",
    "a teacher writing on the board": "una maestra escribiendo en la pizarra",
    "a doctor examining a patient": "un médico examinando a un paciente",
    "a runner crossing the finish line": "un corredor cruzando la línea de meta",
    "a firefighter holding a hose": "un bombero sosteniendo una manguera",
    "a photographer taking pictures": "un fotógrafo tomando fotos",
    "a person meditating by the lake": "una persona meditando junto al lago",
    "a dancer performing on stage": "una bailarina actuando en el escenario",
    "a farmer harvesting wheat": "un agricultor cosechando trigo",
    "a mechanic repairing a car": "un mecánico reparando un coche",
    "a scientist looking into a microscope": "una científica mirando por un microscopio",
    "a barber cutting hair": "un barbero cortando el cabello",
    "a construction worker with a helmet": "un obrero con casco",
    "a person playing the piano": "una persona tocando el piano",
    "a singer performing with a microphone": "una cantante actuando con un micrófono",
    "a student reading a book": "un estudiante leyendo un libro",
    "a child drawing with crayons": "un niño dibujando con crayones",
    "a group of friends laughing": "un grupo de amigos riendo",
    "a person doing yoga": "una persona haciendo yoga",
    "a nurse giving a vaccine": "una enfermera poniendo una vacuna",
    "a man working on a laptop": "un hombre trabajando en un portátil",
    "a waterfall in the jungle": "una cascada en la selva",
    "a snowy mountain peak": "un pico de montaña nevado",
    "a river flowing through the valley": "un río que fluye por el valle",
    "a desert with sand dunes": "un desierto con dunas de arena",
    "a forest in autumn": "un bosque en otoño",
    "waves crashing on the rocks": "olas rompiendo contra las rocas",
    "a rainbow after the storm": "un arco iris después de la tormenta",
    "a night sky full of stars": "un cielo nocturno lleno de estrellas",
    "a calm lake at sunrise": "un lago tranquilo al amanecer",
    "a volcano erupting": "un volcán en erupción",
    "flowers blooming in spring": "flores floreciendo en primavera",
    "a path through the forest": "un sendero a través del bosque",
    "a glacier melting under the sun": "un glaciar derritiéndose bajo el sol",
    "a canyon with red rocks": "un cañón con rocas rojas",
    "a meadow full of daisies": "un prado lleno de margaritas",
    "a lighthouse on the cliff": "un faro en el acantilado",
    "a foggy morning in the mountains": "una mañana con niebla en las montañas",
    "a palm tree on the beach": "una palmera en la playa",
    "a field of sunflowers": "un campo de girasoles",
    "a frozen lake in winter": "un lago congelado en invierno",
    "a vintage camera on a shelf": "una cámara vintage en un estante",
    "a laptop on a wooden desk": "un portátil sobre un escritorio de madera",
    "a smartphone charging": "un teléfono móvil cargando",
    "a pair of headphones on a table": "unos auriculares sobre una mesa",
    "a glowing neon sign": "un cartel de neón brillante",
    "a robot holding a flower": "un robot sosteniendo una flor",
    "a wristwatch made of metal": "un reloj de pulsera de metal",
    "a bookshelf full of books": "una estantería llena de libros",
    "a glass bottle with water": "una botella de vidrio con agua",
    "a lantern lighting the night": "una linterna iluminando la noche",
    "a television in the living room": "un televisor en la sala de estar",
    "a drone flying over a field": "un dron volando sobre un campo",
    "a stack of colorful pillows": "una pila de cojines coloridos",
    "a typewriter with paper": "una máquina de escribir con papel",
    "a 3d printer creating a model": "una impresora 3d creando un modelo",
    "a tablet with a stylus": "una tableta con un lápiz óptico",
    "a pair of glasses on a book": "unas gafas sobre un libro",
    "a vinyl record on a turntable": "un disco de vinilo en un tocadiscos",
    "a candle on a windowsill": "una vela en el alféizar de una ventana",
    "a compass on a map": "una brújula sobre un mapa",
    "a dog catching a frisbee": "un perro atrapando un frisbee",
    "a cat playing with yarn": "un gato jugando con hilo",
    "a horse running in a field": "un caballo corriendo en un campo",
    "a bird sitting on a branch": "un pájaro posado en una rama",
    "a fish swimming in a tank": "un pez nadando en un acuario",
    "a cow grazing in a meadow": "una vaca pastando en un prado",
    "a butterfly landing on a flower": "una mariposa posándose en una flor",
    "a bee collecting nectar": "una abeja recolectando néctar",
    "a lion roaring loudly": "un león rugiendo fuertemente",
    "a penguin walking on ice": "un pingüino caminando sobre el hielo",
    "a turtle swimming in the ocean": "una tortuga nadando en el océano",
    "a parrot with colorful feathers": "un loro con plumas coloridas",
    "a fox hiding behind a tree": "un zorro escondido detrás de un árbol",
    "a squirrel eating a nut": "una ardilla comiendo una nuez",
    "a frog on a leaf": "una rana sobre una hoja",
    "a dog sleeping under a tree": "un perro durmiendo bajo un árbol",
    "a camel walking in the desert": "un camello caminando en el desierto",
    "a bear fishing in a river": "un oso pescando en un río",
    "a cat looking out the window": "un gato mirando por la ventana",
    "a rabbit jumping through the grass": "un conejo saltando entre la hierba",
    "a person smiling happily": "una persona sonriendo felizmente",
    "a man crying in the rain": "un hombre llorando bajo la lluvia",
    "a woman surprised by a gift": "una mujer sorprendida por un regalo",
    "a child laughing loudly": "un niño riendo a carcajadas",
    "a person showing anger": "una persona mostrando enfado",
    "a group celebrating victory": "un grupo celebrando la victoria",
    "a couple hugging each other": "una pareja abrazándose",
    "a person looking worried": "una persona con expresión de preocupación",
    "friends taking a selfie": "amigos tomándose una selfie",
    "a person clapping hands": "una persona aplaudiendo",
    "a person looking at a painting": "una persona mirando una pintura",
    "a crowd cheering at a concert": "una multitud animando en un concierto",
    "a person yawning": "una persona bostezando",
    "a dancer smiling on stage": "una bailarina sonriendo en el escenario",
    "a baby reaching out to touch something": "un bebé extendiendo la mano para tocar algo",
    "a man thinking deeply": "un hombre pensando profundamente",
    "a woman laughing at a joke": "una mujer riendo de un chiste",
    "a child showing excitement": "un niño mostrando entusiasmo",
    "a person meditating peacefully": "una persona meditando en paz",
    "a group enjoying fireworks": "un grupo disfrutando de fuegos artificiales",
    "a street in paris at night": "una calle en parís por la noche",
    "a market full of people": "un mercado lleno de gente",
    "a park with a fountain": "un parque con una fuente",
    "a subway station with lights": "una estación de metro con luces",
    "a quiet village in the hills": "un pueblo tranquilo en las colinas",
    "a busy airport terminal": "una terminal de aeropuerto concurrida",
    "a restaurant by the sea": "un restaurante junto al mar",
    "a library with tall shelves": "una biblioteca con estanterías altas",
    "a stadium full of fans": "un estadio lleno de aficionados",
    "a bridge over the river": "un puente sobre el río",
    "a school classroom with desks": "un aula escolar con escritorios",
    "a museum with ancient statues": "un museo con estatuas antiguas",
    "a hospital corridor": "un pasillo de hospital",
    "a factory producing cars": "una fábrica produciendo coches",
    "a small house with a red roof": "una casa pequeña con techo rojo",
    "a castle surrounded by trees": "un castillo rodeado de árboles",
    "a bus driving through the city": "un autobús conduciendo por la ciudad",
    "a train station with people waiting": "una estación de tren con gente esperando",
    "a cafe on a corner": "una cafetería en una esquina",
    "a boat in the harbor": "un barco en el puerto",
    "a person brushing their teeth": "una persona cepillándose los dientes",
    "a woman tying her shoes": "una mujer atándose los zapatos",
    "a man reading the newspaper": "un hombre leyendo el periódico",
    "a person opening a door": "una persona abriendo una puerta",
    "a student writing notes": "un estudiante tomando apuntes",
    "a person cooking breakfast": "una persona cocinando el desayuno",
    "a mother holding her baby": "una madre sosteniendo a su bebé",
    "a person pouring coffee": "una persona sirviendo café",
    "a person watching television": "una persona viendo la televisión",
    "a family having dinner": "una familia cenando junta",
    "a person washing dishes": "una persona lavando los platos",
    "a man riding a motorcycle": "un hombre montando una motocicleta",
    "a woman talking on the phone": "una mujer hablando por teléfono",
    "a person ironing clothes": "una persona planchando ropa",
    "a child playing with blocks": "un niño jugando con bloques",
    "a person unlocking a bicycle": "una persona desbloqueando una bicicleta",
    "a person lighting a candle": "una persona encendiendo una vela",
    "a person writing a letter": "una persona escribiendo una carta",
    "a man drinking water": "un hombre bebiendo agua",
    "a person folding laundry": "una persona doblando la ropa",
}


test_text_pairs = {
    "a bowl of tomato soup": "un cuenco de sopa de tomate",
    "a plate of fried rice": "un plato de arroz frito",
    "a chocolate ice cream cone": "un cono de helado de chocolate",
    "freshly squeezed lemon juice": "zumo de limón recién exprimido",
    "a sandwich with lettuce and tomato": "un sándwich con lechuga y tomate",

    "a girl riding a skateboard": "una chica montando en monopatín",
    "a man playing chess": "un hombre jugando al ajedrez",
    "a woman walking a dog": "una mujer paseando a un perro",
    "a person holding a map": "una persona sosteniendo un mapa",
    "a group of tourists taking photos": "un grupo de turistas tomando fotos",

    "a mountain reflected in a lake": "una montaña reflejada en un lago",
    "a desert cactus under the sun": "un cactus del desierto bajo el sol",
    "a forest covered in fog": "un bosque cubierto de niebla",
    "a sunrise behind the hills": "un amanecer detrás de las colinas",
    "waves hitting the shore": "olas golpeando la orilla",

    "a digital painting of time": "una pintura digital del tiempo",
    "a clock melting on a table": "un reloj derritiéndose sobre una mesa",
    "a dream inside a mirror": "un sueño dentro de un espejo",
    "a futuristic city in the clouds": "una ciudad futurista en las nubes",
    "a robot reading a book": "un robot leyendo un libro",

    "a cozy living room with a fireplace": "una sala de estar acogedora con chimenea",
    "a train crossing a bridge": "un tren cruzando un puente",
    "a narrow alley with lanterns": "un callejón estrecho con linternas",
    "a small cafe by the river": "una pequeña cafetería junto al río",
    "a classroom full of students": "un aula llena de estudiantes",

    "a puppy running in the grass": "un cachorro corriendo en la hierba",
    "a cat sitting on a chair": "un gato sentado en una silla",
    "a parrot flying through the jungle": "un loro volando por la selva",
    "a dolphin swimming near the surface": "un delfín nadando cerca de la superficie",
    "a panda eating bamboo": "un panda comiendo bambú",

    "a glowing laptop screen in the dark": "una pantalla de portátil brillante en la oscuridad",
    "a car key on a wooden table": "una llave de coche sobre una mesa de madera",
    "a camera on a tripod": "una cámara en un trípode",
    "a smartwatch showing the time": "un reloj inteligente mostrando la hora",
    "a pair of sneakers on the floor": "un par de zapatillas en el suelo",

    "a person smiling while dancing": "una persona sonriendo mientras baila",
    "a couple watching the sunset": "una pareja viendo la puesta de sol",
    "a man shouting at the sky": "un hombre gritando al cielo",
    "a child hugging a teddy bear": "un niño abrazando un oso de peluche",
    "a woman whispering a secret": "una mujer susurrando un secreto",

    "a person crossing the street": "una persona cruzando la calle",
    "a worker cleaning the floor": "un trabajador limpiando el suelo",
    "a cyclist waiting at a red light": "un ciclista esperando en un semáforo rojo",
    "a person shopping for groceries": "una persona comprando comestibles",
    "a man opening an umbrella": "un hombre abriendo un paraguas",

    "a storm forming over the ocean": "una tormenta formándose sobre el océano",
    "a rainbow over the waterfall": "un arco iris sobre la cascada",
    "a street filled with autumn leaves": "una calle llena de hojas de otoño",
    "a lighthouse shining at night": "un faro brillando por la noche",
    "a balloon floating in the sky": "un globo flotando en el cielo"
}




In [8]:
class eng_to_span_dataset(Dataset):
  def __init__(self, eng_span_dict):
    self.eng_span_dict = eng_span_dict
    self.eng_texts = list(eng_span_dict.keys())
    self.span_texts = list(eng_span_dict.values())

  def __len__(self):
    return len(self.eng_texts)

  def __getitem__(self, idx):
    return self.eng_texts[idx], self.span_texts[idx]


In [9]:
train_dataset = eng_to_span_dataset(text_pairs)
test_dataset = eng_to_span_dataset(test_text_pairs)
train_loader = DataLoader(train_dataset, batch_size=24, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
adapter = CLIP2SDAdapter(in_dim=1024, out_dim=1024, in_tokens=77, out_tokens=77).to(device)

epochs = 50
optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-4)
criterion = nn.MSELoss()

best_val_loss = float('inf')
early_stop_patience = 10
no_val_improvement_counter = 0

mclip_model.eval()
sd_text_encoder.eval()
adapter.train()
for epoch in range(epochs):
    total_train_loss = 0
    total_val_loss = 0
    for batch in train_loader:
      eng_cap, span_cap = batch

      m_embs = mclip_model.get_last_hidden_state(span_cap, mclip_tokenizer, device).to(device)

      sd_tok = sd_tokenizer(eng_cap, padding="max_length", truncation=True, max_length=77, return_tensors="pt").to(device)
      with torch.no_grad():
          sd_embs = sd_text_encoder(sd_tok.input_ids)[0].to(device)

      out = adapter(m_embs)
      loss = criterion(out, sd_embs.float())

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_train_loss += loss.item()


    adapter.eval()
    for batch in test_loader:
      eng_cap, span_cap = batch

      m_embs = mclip_model.get_last_hidden_state(span_cap, mclip_tokenizer, device).to(device)
      sd_tok = sd_tokenizer(eng_cap, padding="max_length", truncation=True, max_length=77, return_tensors="pt").to(device)
      with torch.no_grad():
          sd_embs = sd_text_encoder(sd_tok.input_ids)[0].to(device)

      out = adapter(m_embs)
      loss = criterion(out, sd_embs.float())
      total_val_loss += loss.item()

    if total_val_loss < best_val_loss:
      best_val_loss = total_val_loss
      torch.save(adapter.state_dict(), f"ep{epoch}_best_adapter.pt")
      no_val_improvement_counter = 0
    else:
      no_val_improvement_counter+=1
      if no_val_improvement_counter >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch}")
        break


    print(f"Epoch {epoch+1}, Train Loss: {total_train_loss / len(train_loader):.6f}, Val Loss: {total_val_loss / len(test_loader):.6f}")

Epoch 1, Train Loss: 1.132932, Val Loss: 1.117470
Epoch 2, Train Loss: 1.106085, Val Loss: 1.097837
Epoch 3, Train Loss: 1.088281, Val Loss: 1.081219


In [ ]:
#https://en.wikipedia.org/wiki/List_of_Mexican_dishes
texts = [
   'tacos de pescado',
   'Tortilla española',
   'Gambas al ajillo',
   "tacos al pastor",
   "English Breakfast"]

embeddings = mclip_model.forward(texts, mclip_tokenizer)
last_layer_embeds = mclip_model.get_last_hidden_state(texts, mclip_tokenizer)
print(embeddings.shape)
print(last_layer_embeds.shape)

In [ ]:
img_cap_pairs = {}

for text, embeds in zip(texts, last_layer_embeds):
    print(text)
    if embeds.ndim == 2:
        embeds = embeds.unsqueeze(0)
    print(embeds.shape)
    gen_img = sd_model(prompt_embeds=embeds)
    img_cap_pairs[text] = gen_img[0][0]
    img_cap_pairs[text].save(f"{text}.png")